In [56]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import re
import json
import time
import torch
import subprocess
import torch.nn as nn
from collections import defaultdict
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoConfig, AutoModel
from torchcrf import CRF

# ==========================================
# CONFIGURATION
# ==========================================
MODEL_DIR = "./xlm_roberta_large_checkpointV2"
LOG_FILE = "address_split_results_xlm_large.log"
TEST_FILE = "data/test.jsonl"
MAX_LEN = 128
BATCH_SIZE = 32  # Lowered for XLM-R Large

# ==========================================
# CUSTOM ARCHITECTURE (Updated for XLM-R Large + Constraints)
# ==========================================
class BertCRFForTokenClassification(torch.nn.Module):
    def __init__(self, config, model_name_or_path, label_list):
        super().__init__()
        self.config = config
        self.label_list = label_list

        self.bert = AutoModel.from_pretrained(model_name_or_path, config=config, ignore_mismatched_sizes=True)
        self.dropout = torch.nn.Dropout(config.hidden_dropout_prob)
        self.classifier = torch.nn.Linear(config.hidden_size, config.num_labels)
        self.crf = CRF(num_tags=config.num_labels, batch_first=True)

        self._set_bio_constraints(label_list)

    def _set_bio_constraints(self, label_list):
        num_tags = len(label_list)
        self.crf.transitions.data.fill_(-1e4)
        self.crf.start_transitions.data.fill_(-1e4)
        self.crf.end_transitions.data.fill_(-1e4)

        id2label = {i: l for i, l in enumerate(label_list)}
        label2id = {l: i for i, l in id2label.items()}

        o_id = label2id["O"]
        self.crf.transitions.data[o_id, :] = 0.0
        self.crf.transitions.data[:, o_id] = 0.0
        self.crf.start_transitions.data[o_id] = 0.0
        self.crf.end_transitions.data[o_id] = 0.0

        for tag in label_list:
            if tag == "O": continue
            tid = label2id[tag]
            self.crf.start_transitions.data[tid] = 0.0
            self.crf.end_transitions.data[tid] = 0.0

            if tag.startswith("B-"):
                itype = "I-" + tag[2:]
                if itype in label2id:
                    self.crf.transitions.data[tid, label2id[itype]] = 0.0
                for other in label_list:
                    if other.startswith("B-"):
                        self.crf.transitions.data[tid, label2id[other]] = 0.0
            elif tag.startswith("I-"):
                self.crf.transitions.data[tid, tid] = 0.0
                for other in label_list:
                    if other.startswith("B-"):
                        self.crf.transitions.data[tid, label2id[other]] = 0.0

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs[0])
        emissions = self.classifier(sequence_output)
        mask = attention_mask.type(torch.uint8)

        if labels is not None:
            safe_labels = torch.where(labels >= 0, labels, torch.zeros_like(labels))
            return -self.crf(emissions, tags=safe_labels, mask=mask, reduction="mean")
        else:
            tags = self.crf.decode(emissions, mask=mask)
            return tags, emissions

# ==========================================
# ADDRESS PARSER CLASS
# ==========================================
class HKAddressParser:
    def __init__(self, model_path, max_len=128):
        self.model_path = model_path
        self.max_len = max_len
        self.device = self._get_emptiest_gpu_safely()
        print(f"DEBUG: Using Device -> {self.device}")

        self.config = AutoConfig.from_pretrained(model_path)
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        
        # Dynamically build label list for constraints
        label_list = [self.config.id2label[k] if isinstance(k, int) else self.config.id2label[str(k)] 
                      for k in sorted([int(k) for k in self.config.id2label.keys()])]

        self.model = BertCRFForTokenClassification(self.config, model_path, label_list)

        weights_path = os.path.join(model_path, "pytorch_model.bin")
        if os.path.exists(weights_path):
            self._load_weights_with_progress(weights_path)
        else:
            st_path = os.path.join(model_path, "model.safetensors")
            if os.path.exists(st_path):
                self._load_weights_with_progress(st_path, safetensors=True)
            else:
                print(f"⚠️ Warning: Weights not found at {weights_path}")

        self.model.to(self.device)
        self.model.eval()

    def _load_weights_with_progress(self, weights_path, safetensors=False):
        print(f"📦 Loading weights from {weights_path} ...")
        if safetensors:
            from safetensors.torch import load_file
            state_dict = load_file(weights_path, device="cpu")
        else:
            state_dict = torch.load(weights_path, map_location="cpu")

        model_keys = set(self.model.state_dict().keys())
        loaded = 0
        missing_in_ckpt = []
        with tqdm(total=len(model_keys), desc="Loading weights", unit="tensor") as pbar:
            own = self.model.state_dict()
            for name in list(own.keys()):
                if name in state_dict and own[name].shape == state_dict[name].shape:
                    own[name].copy_(state_dict[name])
                    loaded += 1
                else:
                    missing_in_ckpt.append(name)
                pbar.update(1)
                pbar.set_postfix(loaded=loaded)

        unexpected = [k for k in state_dict.keys() if k not in model_keys]
        print(
            f"✅ Loaded {loaded}/{len(model_keys)} tensors | "
            f"missing={len(missing_in_ckpt)} unexpected={len(unexpected)}"
        )
        if missing_in_ckpt[:5]:
            print(f" e.g. missing: {missing_in_ckpt[:5]}")
        if unexpected[:5]:
            print(f" e.g. unexpected: {unexpected[:5]}")

    @staticmethod
    def _get_emptiest_gpu_safely():
        if not torch.cuda.is_available():
            return torch.device("cpu")
        try:
            print("\n🔍 Scanning available GPUs safely via nvidia-smi...")
            result = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=index,memory.free,utilization.gpu",
                 "--format=csv,nounits,noheader"],
                encoding="utf-8"
            )
            best_id = -1
            max_free_mb = 0
            fallback_id = 0
            fallback_max_mb = 0

            for line in result.strip().split("\n"):
                parts = line.split(", ")
                gpu_id = int(parts[0])
                free_memory = int(parts[1])
                gpu_util = int(parts[2])
                print(f" GPU {gpu_id}: {free_memory} MB free | {gpu_util}% util")

                if free_memory > fallback_max_mb:
                    fallback_max_mb = free_memory
                    fallback_id = gpu_id

                if gpu_util < 30:
                    if free_memory > max_free_mb:
                        max_free_mb = free_memory
                        best_id = gpu_id

            if best_id != -1:
                print(f"--> Selected GPU {best_id} with {max_free_mb} MB free VRAM and low utilization.\n")
                return torch.device(f"cuda:{best_id}")
            else:
                print(f"⚠️ All GPUs are highly utilized (>= 30%). Falling back to GPU {fallback_id} with {fallback_max_mb} MB free.\n")
                return torch.device(f"cuda:{fallback_id}")
        except Exception as e:
            print(f"⚠️ Failed to query nvidia-smi: {e}. Falling back to cuda:0.")
            return torch.device("cuda:0")

    @staticmethod
    def _extract_3d_components(parsed_entities):
        components = defaultdict(list)
        confs = defaultdict(list)

        for entity in parsed_entities:
            tag = entity["entity_group"]
            if tag == "O":
                continue
            components[tag].append(entity["word"])
            confs[tag].append(entity["conf"])

        formatted_output = {}
        conf_output = {}
        for tag, words in components.items():
            formatted_output[tag] = "".join(words)
            conf_output[tag] = sum(confs[tag]) / len(confs[tag]) if confs[tag] else 0.0

        return formatted_output, conf_output

    @staticmethod
    def _map_extracted_to_standard_keys(extracted_data, conf_data=None):
        mapping = {
            "UNIT": "flat", "FLOOR": "floor", "BLOCK": "block", "PHASE": "phase",
            "BUILDING_NAME": "building_name", "ESTATE_NAME": "estate_name",
            "VILLAGE_NAME": "village_name", "BUILDING_NUMBER": "building_number",
            "STREET_NAME": "street_name", "SUB_DISTRICT": "sub_district",
            "DISTRICT": "district", "REGION": "region"
        }
        mapped = {mapping.get(k, k.lower()): v for k, v in extracted_data.items()}
        if conf_data is not None:
            mapped_conf = {mapping.get(k, k.lower()): v for k, v in conf_data.items()}
            return mapped, mapped_conf
        return mapped

    def _compute_situation_aware_confidence(self, extracted_mapped, conf_mapped):
        exclude = {"district", "region", "sub_district"}
        core = set()

        if "flat" in extracted_mapped: core.add("flat")
        if "floor" in extracted_mapped: core.add("floor")

        if "estate_name" in extracted_mapped:
            core.add("estate_name")
            if "block" in extracted_mapped: core.add("block")
            if "building_name" in extracted_mapped: core.add("building_name")
            if "phase" in extracted_mapped: core.add("phase")
        elif "building_name" in extracted_mapped and "block" in extracted_mapped:
            core.add("block")
            core.add("building_name")
        elif "building_name" in extracted_mapped:
            core.add("building_name")
        else:
            if "block" in extracted_mapped:
                core.add("block")
            elif "street_name" in extracted_mapped:
                core.add("street_name")
                if "building_number" in extracted_mapped: core.add("building_number")
            elif "village_name" in extracted_mapped:
                core.add("village_name")
                if "building_number" in extracted_mapped: core.add("building_number")

        overall = 1.0
        used = []
        for k in core:
            if k in conf_mapped and k not in exclude and conf_mapped[k] > 0:
                overall *= conf_mapped[k]
                used.append(k)

        if not used:
            for k, c in conf_mapped.items():
                if k not in exclude and c > 0:
                    overall *= c
                    used.append(k)

        return overall, used

    @staticmethod
    def _split_address(extracted_labels, original_input, parsed_entities):
        line1_keys = set()
        
        if "flat" in extracted_labels: line1_keys.add("flat")
        if "floor" in extracted_labels: line1_keys.add("floor")
        
        if "estate_name" in extracted_labels:
            if "block" in extracted_labels: line1_keys.add("block")
            if "building_name" in extracted_labels: line1_keys.add("building_name")
            if "phase" in extracted_labels: line1_keys.add("phase")
        elif "building_name" in extracted_labels and "block" in extracted_labels:
            line1_keys.add("block")
            line1_keys.add("building_name")
        elif "building_name" in extracted_labels:
            line1_keys.add("building_name")
        else:
            if "block" in extracted_labels:
                line1_keys.add("block")
            elif "street_name" in extracted_labels:
                if "building_number" in extracted_labels: line1_keys.add("building_number")
                line1_keys.add("street_name")
            elif "village_name" in extracted_labels:
                if "building_number" in extracted_labels: line1_keys.add("building_number")
                line1_keys.add("village_name")

        is_chinese = any("\u4e00" <= char <= "\u9fff" for char in original_input)
        en_order = {
            "flat": 1, "floor": 1, "block": 2, "phase": 2,
            "building_name": 3, "estate_name": 3,
            "building_number": 4, "street_name": 4, "village_name": 4,
            "sub_district": 5, "district": 5, "region": 5
        }
        zh_order = {
            "region": 1, "district": 1, "sub_district": 1, 
            "estate_name": 2, "village_name": 2,
            "street_name": 3, "building_number": 3, 
            "phase": 4, "building_name": 4,
            "block": 5, "floor": 6, "flat": 6
        }
        order_map = zh_order if is_chinese else en_order

        def sort_items(items):
            items.sort(key=lambda x: (
                order_map.get(x[0], 99),
                original_input.find(x[1]) if original_input.find(x[1]) != -1 else 999
            ))
            return [val for key, val in items]

        def assemble_line(values):
            if not values: return ""
            result = values[0]
            for i in range(1, len(values)):
                clean_prev = result.strip()
                clean_curr = values[i].strip()
                prev_char = clean_prev[-1] if clean_prev else ""
                curr_char = clean_curr[0] if clean_curr else ""
                is_prev_alpha = bool(re.match(r"[A-Za-z0-9]", prev_char))
                is_curr_alpha = bool(re.match(r"[A-Za-z0-9]", curr_char))
                has_space = result.endswith(" ") or values[i].startswith(" ")
                
                if has_space: result += values[i]
                elif is_prev_alpha and is_curr_alpha: result += " " + values[i]
                elif not is_chinese: result += " " + values[i]
                elif is_prev_alpha != is_curr_alpha: result += " " + values[i]
                else: result += values[i]
            return re.sub(r"\s+", " ", result).strip()

        micro_items = [(k, extracted_labels[k]) for k in line1_keys]
        micro_string = assemble_line(sort_items(micro_items))

        mapping = {
            "UNIT": "flat", "FLOOR": "floor", "BLOCK": "block", "PHASE": "phase",
            "BUILDING_NAME": "building_name", "ESTATE_NAME": "estate_name",
            "VILLAGE_NAME": "village_name", "BUILDING_NUMBER": "building_number",
            "STREET_NAME": "street_name", "SUB_DISTRICT": "sub_district",
            "DISTRICT": "district", "REGION": "region"
        }

        macro_tokens = []
        for entity in parsed_entities:
            raw_tag = entity["entity_group"]
            mapped_tag = mapping.get(raw_tag, raw_tag.lower())
            if mapped_tag not in line1_keys:
                macro_tokens.append(entity["word"])

        macro_string = "".join(macro_tokens)
        macro_string = re.sub(r"\s+", " ", macro_string).strip()
        macro_string = re.sub(r"\s*[,/\\-]\s*,\s*", ", ", macro_string)
        macro_string = re.sub(r"\s*,\s*[,/\\-]\s*", ", ", macro_string)
        macro_string = re.sub(r"\s*,\s*,\s*", ", ", macro_string)
        macro_string = re.sub(r"^[,/\\\-\s]+|[,/\\\-\s]+$", "", macro_string)

        if is_chinese: return {"line1": macro_string, "line2": micro_string}
        else: return {"line1": micro_string, "line2": macro_string}

    def parse_batch(self, address_pairs, batch_size=32, show_progress=True):
        all_results = []
        iterator = range(0, len(address_pairs), batch_size)
        if show_progress:
            iterator = tqdm(iterator, desc="Processing Batches",
                            total=(len(address_pairs) + batch_size - 1) // batch_size)

        for i in iterator:
            batch = address_pairs[i : i + batch_size]
            full_addresses = []

            for p1, p2 in batch:
                parts = [p.strip() for p in (p1, p2) if p.strip()]
                full_addresses.append(" ".join(parts) if parts else "")

            encoded = self.tokenizer(
                full_addresses,
                padding=True,
                truncation=True,
                max_length=self.max_len,
                return_offsets_mapping=True,
                return_tensors="pt"
            )

            input_ids = encoded["input_ids"].to(self.device)
            attention_mask = encoded["attention_mask"].to(self.device)
            batch_offsets = encoded["offset_mapping"].tolist()

            try:
                with torch.no_grad():
                    prediction_ids_batch, emissions_batch = self.model(
                        input_ids=input_ids, attention_mask=attention_mask
                    )
            except Exception as e:
                all_results.extend([("", "", {}, {}, 0.0, f"ERROR: {str(e)}")] * len(batch))
                continue

            for idx, address_str in enumerate(full_addresses):
                if not address_str:
                    all_results.append(("", "", {}, {}, 0.0, "EMPTY_INPUT"))
                    continue

                char_tags = ["O"] * len(address_str)
                char_confs = [0.0] * len(address_str)
                offsets = batch_offsets[idx]
                prediction_ids = prediction_ids_batch[idx]
                token_probs = torch.softmax(emissions_batch[idx], dim=-1)

                for j, tag_id in enumerate(prediction_ids):
                    start, end = offsets[j]
                    if start == end: continue

                    if isinstance(tag_id, int):
                        tag = self.config.id2label[tag_id]
                        conf = token_probs[j, tag_id].item()
                    else:
                        tag = self.config.id2label[str(tag_id)]
                        conf = token_probs[j, int(tag_id)].item()

                    for c in range(start, end):
                        if char_tags[c] == "O":
                            char_tags[c] = tag
                            char_confs[c] = conf

                parsed_entities = []
                for match in re.finditer(r"([a-zA-Z0-9]+|[\u4e00-\u9fff]|[^\s])(\s*)", address_str):
                    token_str = match.group(1)
                    trailing_space = match.group(2)
                    start_idx = match.start(1)

                    tag = char_tags[start_idx]
                    conf = char_confs[start_idx]
                    entity_group = tag.replace("B-", "").replace("I-", "")
                    if tag == "O": entity_group = "O"

                    parsed_entities.append({
                        "entity_group": entity_group,
                        "word": token_str + trailing_space,
                        "conf": conf
                    })

                extracted_raw, conf_raw = self._extract_3d_components(parsed_entities)
                extracted_mapped, conf_mapped = self._map_extracted_to_standard_keys(extracted_raw, conf_raw)
                overall_conf, used_cores = self._compute_situation_aware_confidence(extracted_mapped, conf_mapped)
                split_result = self._split_address(extracted_mapped, address_str, parsed_entities)

                all_results.append((
                    split_result["line1"],
                    split_result["line2"],
                    extracted_mapped,
                    conf_mapped,
                    overall_conf,
                    "SUCCESS"
                ))

        return all_results

def main():
    print(f"DEBUG: Initializing Parser from {MODEL_DIR}...")

    if not os.path.exists(MODEL_DIR):
        print(f"❌ Error: {MODEL_DIR} not found.")
        return

    if not os.path.exists(TEST_FILE):
        print(f"❌ Error: Dataset file '{TEST_FILE}' not found.")
        return

    parser = HKAddressParser(model_path=MODEL_DIR, max_len=MAX_LEN)

    print(f"🚀 Loading dataset from {TEST_FILE}...")
    with open(TEST_FILE, "r", encoding="utf-8") as file:
        raw_lines = [line for line in file if line.strip() and not line.startswith("#")]

    address_pairs = []
    for line in raw_lines:
        data = json.loads(line)
        address_pairs.append((data["input"].strip(), ""))

    print(f"🚀 Running model inference on {len(address_pairs)} addresses...")
    start_time = time.perf_counter()

    results = parser.parse_batch(address_pairs, batch_size=BATCH_SIZE, show_progress=True)

    print(f"✍️ Writing results to {LOG_FILE}...")
    with open(LOG_FILE, "w", encoding="utf-8") as log:
        for idx, (original_input_pair, result) in enumerate(zip(address_pairs, results)):
            address = original_input_pair[0]
            l1, l2, tags, confs, overall, status = result
            conf_str = {k: round(v, 4) for k, v in confs.items()}

            log_output = (
                f"--- Result {idx + 1} ---\n"
                f"Original Input : {address}\n"
                f"Extracted Tags : {tags}\n"
                f"Per-tag Confs  : {conf_str}\n"
                f"Overall Conf   : {overall:.6f}  (situation-aware product of core tags)\n"
                f"Output Line 1  : {l1}\n"
                f"Output Line 2  : {l2}\n"
                f"Status         : {status}\n"
                f"{'-'*50}\n"
            )
            log.write(log_output)

    total_time = time.perf_counter() - start_time
    print("\n" + "=" * 40)
    print("✅ PROCESSING COMPLETE")
    print("=" * 40)
    print(f"Addresses Processed : {len(address_pairs)}")
    print(f"Batch Size          : {BATCH_SIZE}")
    print(f"Results saved to    : {LOG_FILE}")
    print(f"Total Runtime       : {total_time:.4f} seconds")

if __name__ == "__main__":
    main()

DEBUG: Initializing Parser from ./bert_crf_model...

🔍 Scanning available GPUs safely via nvidia-smi...
 GPU 0: 14613 MB free | 0% util
 GPU 1: 5396 MB free | 100% util
 GPU 2: 14696 MB free | 0% util
 GPU 3: 7796 MB free | 0% util
--> Selected GPU 2 with 14696 MB free VRAM and low utilization.

DEBUG: Using Device -> cuda:2
📦 Loading weights from ./bert_crf_model/pytorch_model.bin ...


Loading weights:   0%|          | 0/204 [00:00<?, ?tensor/s]

✅ Loaded 204/204 tensors | missing=0 unexpected=0
🚀 Loading dataset from bilstm_data/bilstm_test.jsonl...
🚀 Running model inference on 20814 addresses...


Processing Batches:   0%|          | 0/326 [00:00<?, ?it/s]

✍️ Writing results to address_split_results.log...

✅ PROCESSING COMPLETE
Addresses Processed : 20814
Batch Size          : 64
Results saved to    : address_split_results.log
Total Runtime       : 63.1381 seconds


In [4]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import re
import torch
import subprocess
import torch.nn as nn
from collections import defaultdict
from tqdm import tqdm
from transformers import AutoTokenizer, AutoConfig, AutoModel
from torchcrf import CRF

# ==========================================
# CUSTOM ARCHITECTURE (Updated for XLM-R Large + Constraints)
# ==========================================
class BertCRFForTokenClassification(torch.nn.Module):
    def __init__(self, config, model_name_or_path, label_list):
        super().__init__()
        self.config = config
        self.label_list = label_list

        self.bert = AutoModel.from_pretrained(model_name_or_path, config=config, ignore_mismatched_sizes=True)
        self.dropout = torch.nn.Dropout(config.hidden_dropout_prob)
        self.classifier = torch.nn.Linear(config.hidden_size, config.num_labels)
        self.crf = CRF(num_tags=config.num_labels, batch_first=True)

        self._set_bio_constraints(label_list)

    def _set_bio_constraints(self, label_list):
        num_tags = len(label_list)
        self.crf.transitions.data.fill_(-1e4)
        self.crf.start_transitions.data.fill_(-1e4)
        self.crf.end_transitions.data.fill_(-1e4)

        id2label = {i: l for i, l in enumerate(label_list)}
        label2id = {l: i for i, l in id2label.items()}

        o_id = label2id["O"]
        self.crf.transitions.data[o_id, :] = 0.0
        self.crf.transitions.data[:, o_id] = 0.0
        self.crf.start_transitions.data[o_id] = 0.0
        self.crf.end_transitions.data[o_id] = 0.0

        for tag in label_list:
            if tag == "O": continue
            tid = label2id[tag]
            self.crf.start_transitions.data[tid] = 0.0
            self.crf.end_transitions.data[tid] = 0.0

            if tag.startswith("B-"):
                itype = "I-" + tag[2:]
                if itype in label2id:
                    self.crf.transitions.data[tid, label2id[itype]] = 0.0
                for other in label_list:
                    if other.startswith("B-"):
                        self.crf.transitions.data[tid, label2id[other]] = 0.0
            elif tag.startswith("I-"):
                self.crf.transitions.data[tid, tid] = 0.0
                for other in label_list:
                    if other.startswith("B-"):
                        self.crf.transitions.data[tid, label2id[other]] = 0.0

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs[0])
        emissions = self.classifier(sequence_output)
        mask = attention_mask.type(torch.uint8)

        if labels is not None:
            safe_labels = torch.where(labels >= 0, labels, torch.zeros_like(labels))
            return -self.crf(emissions, tags=safe_labels, mask=mask, reduction="mean")
        else:
            tags = self.crf.decode(emissions, mask=mask)
            return tags, emissions

# ==========================================
# ADDRESS PARSER CLASS
# ==========================================
class HKAddressParser:
    def __init__(self, model_path, max_len=128):
        self.model_path = model_path
        self.max_len = max_len
        self.device = self._get_emptiest_gpu_safely()
        print(f"DEBUG: Using Device -> {self.device}")

        self.config = AutoConfig.from_pretrained(model_path)
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)

        # Dynamically build label list for constraints
        label_list = [self.config.id2label[k] if isinstance(k, int) else self.config.id2label[str(k)] 
                      for k in sorted([int(k) for k in self.config.id2label.keys()])]

        self.model = BertCRFForTokenClassification(self.config, model_path, label_list)

        weights_path = os.path.join(model_path, "pytorch_model.bin")
        if os.path.exists(weights_path):
            self._load_weights_with_progress(weights_path)
        else:
            st_path = os.path.join(model_path, "model.safetensors")
            if os.path.exists(st_path):
                self._load_weights_with_progress(st_path, safetensors=True)
            else:
                print(f"⚠️ Warning: Weights not found at {weights_path}")

        self.model.to(self.device)
        self.model.eval()

    def _load_weights_with_progress(self, weights_path, safetensors=False):
        print(f"📦 Loading weights from {weights_path} ...")
        if safetensors:
            from safetensors.torch import load_file
            state_dict = load_file(weights_path, device="cpu")
        else:
            state_dict = torch.load(weights_path, map_location="cpu")

        model_keys = set(self.model.state_dict().keys())
        loaded = 0
        missing_in_ckpt = []
        with tqdm(total=len(model_keys), desc="Loading weights", unit="tensor") as pbar:
            own = self.model.state_dict()
            for name in list(own.keys()):
                if name in state_dict and own[name].shape == state_dict[name].shape:
                    own[name].copy_(state_dict[name])
                    loaded += 1
                else:
                    missing_in_ckpt.append(name)
                pbar.update(1)
                pbar.set_postfix(loaded=loaded)

        unexpected = [k for k in state_dict.keys() if k not in model_keys]
        print(
            f"✅ Loaded {loaded}/{len(model_keys)} tensors | "
            f"missing={len(missing_in_ckpt)} unexpected={len(unexpected)}"
        )

    @staticmethod
    def _get_emptiest_gpu_safely():
        if not torch.cuda.is_available():
            return torch.device("cpu")
        try:
            print("\n🔍 Scanning available GPUs safely via nvidia-smi...")
            result = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=index,memory.free,utilization.gpu",
                 "--format=csv,nounits,noheader"],
                encoding="utf-8"
            )
            best_id = -1
            max_free_mb = 0
            fallback_id = 0
            fallback_max_mb = 0

            for line in result.strip().split("\n"):
                parts = line.split(", ")
                gpu_id = int(parts[0])
                free_memory = int(parts[1])
                gpu_util = int(parts[2])

                if free_memory > fallback_max_mb:
                    fallback_max_mb = free_memory
                    fallback_id = gpu_id

                if gpu_util < 30:
                    if free_memory > max_free_mb:
                        max_free_mb = free_memory
                        best_id = gpu_id

            if best_id != -1:
                return torch.device(f"cuda:{best_id}")
            else:
                return torch.device(f"cuda:{fallback_id}")
        except Exception as e:
            return torch.device("cuda:0")

    @staticmethod
    def _extract_3d_components(parsed_entities):
        components = defaultdict(list)
        confs = defaultdict(list)
        for entity in parsed_entities:
            tag = entity["entity_group"]
            if tag == "O": continue
            components[tag].append(entity["word"])
            confs[tag].append(entity["conf"])

        formatted_output = {}
        conf_output = {}
        for tag, words in components.items():
            formatted_output[tag] = "".join(words)
            conf_output[tag] = sum(confs[tag]) / len(confs[tag]) if confs[tag] else 0.0

        return formatted_output, conf_output

    @staticmethod
    def _map_extracted_to_standard_keys(extracted_data, conf_data=None):
        mapping = {
            "UNIT": "flat", "FLOOR": "floor", "BLOCK": "block", "PHASE": "phase",
            "BUILDING_NAME": "building_name", "ESTATE_NAME": "estate_name",
            "VILLAGE_NAME": "village_name", "BUILDING_NUMBER": "building_number",
            "STREET_NAME": "street_name", "SUB_DISTRICT": "sub_district",
            "DISTRICT": "district", "REGION": "region"
        }
        mapped = {mapping.get(k, k.lower()): v for k, v in extracted_data.items()}
        if conf_data is not None:
            mapped_conf = {mapping.get(k, k.lower()): v for k, v in conf_data.items()}
            return mapped, mapped_conf
        return mapped

    def _compute_situation_aware_confidence(self, extracted_mapped, conf_mapped):
        exclude = {"district", "region", "sub_district"}
        core = set()

        if "flat" in extracted_mapped: core.add("flat")
        if "floor" in extracted_mapped: core.add("floor")

        if "estate_name" in extracted_mapped:
            core.add("estate_name")
            if "block" in extracted_mapped: core.add("block")
            if "building_name" in extracted_mapped: core.add("building_name")
            if "phase" in extracted_mapped: core.add("phase")
        elif "building_name" in extracted_mapped and "block" in extracted_mapped:
            core.add("block")
            core.add("building_name")
        elif "building_name" in extracted_mapped:
            core.add("building_name")
        else:
            if "block" in extracted_mapped:
                core.add("block")
            elif "street_name" in extracted_mapped:
                core.add("street_name")
                if "building_number" in extracted_mapped: core.add("building_number")
            elif "village_name" in extracted_mapped:
                core.add("village_name")
                if "building_number" in extracted_mapped: core.add("building_number")

        overall = 1.0
        used = []
        for k in core:
            if k in conf_mapped and k not in exclude and conf_mapped[k] > 0:
                overall *= conf_mapped[k]
                used.append(k)

        if not used:
            for k, c in conf_mapped.items():
                if k not in exclude and c > 0:
                    overall *= c
                    used.append(k)

        return overall, used

    @staticmethod
    def _split_address(extracted_labels, original_input, parsed_entities):
        line1_keys = set()
        
        if "flat" in extracted_labels: line1_keys.add("flat")
        if "floor" in extracted_labels: line1_keys.add("floor")
        
        if "estate_name" in extracted_labels:
            if "block" in extracted_labels: line1_keys.add("block")
            if "building_name" in extracted_labels: line1_keys.add("building_name")
            if "phase" in extracted_labels: line1_keys.add("phase")
        elif "building_name" in extracted_labels and "block" in extracted_labels:
            line1_keys.add("block")
            line1_keys.add("building_name")
        elif "building_name" in extracted_labels:
            line1_keys.add("building_name")
        else:
            if "block" in extracted_labels:
                line1_keys.add("block")
            elif "street_name" in extracted_labels:
                if "building_number" in extracted_labels: line1_keys.add("building_number")
                line1_keys.add("street_name")
            elif "village_name" in extracted_labels:
                if "building_number" in extracted_labels: line1_keys.add("building_number")
                line1_keys.add("village_name")

        is_chinese = any("\u4e00" <= char <= "\u9fff" for char in original_input)
        en_order = {
            "flat": 1, "floor": 1, "block": 2, "phase": 2,
            "building_name": 3, "estate_name": 3,
            "building_number": 4, "street_name": 4, "village_name": 4,
            "sub_district": 5, "district": 5, "region": 5
        }
        zh_order = {
            "region": 1, "district": 1, "sub_district": 1, 
            "estate_name": 2, "village_name": 2,
            "street_name": 3, "building_number": 3, 
            "phase": 4, "building_name": 4,
            "block": 5, "floor": 6, "flat": 6
        }
        order_map = zh_order if is_chinese else en_order

        def sort_items(items):
            items.sort(key=lambda x: (
                order_map.get(x[0], 99),
                original_input.find(x[1]) if original_input.find(x[1]) != -1 else 999
            ))
            return [val for key, val in items]

        def assemble_line(values):
            if not values: return ""
            result = values[0]
            for i in range(1, len(values)):
                clean_prev = result.strip()
                clean_curr = values[i].strip()
                prev_char = clean_prev[-1] if clean_prev else ""
                curr_char = clean_curr[0] if clean_curr else ""
                is_prev_alpha = bool(re.match(r"[A-Za-z0-9]", prev_char))
                is_curr_alpha = bool(re.match(r"[A-Za-z0-9]", curr_char))
                has_space = result.endswith(" ") or values[i].startswith(" ")
                
                if has_space: result += values[i]
                elif is_prev_alpha and is_curr_alpha: result += " " + values[i]
                elif not is_chinese: result += " " + values[i]
                elif is_prev_alpha != is_curr_alpha: result += " " + values[i]
                else: result += values[i]
            return re.sub(r"\s+", " ", result).strip()

        micro_items = [(k, extracted_labels[k]) for k in line1_keys]
        micro_string = assemble_line(sort_items(micro_items))

        mapping = {
            "UNIT": "flat", "FLOOR": "floor", "BLOCK": "block", "PHASE": "phase",
            "BUILDING_NAME": "building_name", "ESTATE_NAME": "estate_name",
            "VILLAGE_NAME": "village_name", "BUILDING_NUMBER": "building_number",
            "STREET_NAME": "street_name", "SUB_DISTRICT": "sub_district",
            "DISTRICT": "district", "REGION": "region"
        }

        macro_tokens = []
        for entity in parsed_entities:
            raw_tag = entity["entity_group"]
            mapped_tag = mapping.get(raw_tag, raw_tag.lower())
            if mapped_tag not in line1_keys:
                macro_tokens.append(entity["word"])

        macro_string = "".join(macro_tokens)
        macro_string = re.sub(r"\s+", " ", macro_string).strip()
        macro_string = re.sub(r"\s*[,/\\-]\s*,\s*", ", ", macro_string)
        macro_string = re.sub(r"\s*,\s*[,/\\-]\s*", ", ", macro_string)
        macro_string = re.sub(r"\s*,\s*,\s*", ", ", macro_string)
        macro_string = re.sub(r"^[,/\\\-\s]+|[,/\\\-\s]+$", "", macro_string)

        if is_chinese: return {"line1": macro_string, "line2": micro_string}
        else: return {"line1": micro_string, "line2": macro_string}

    def parse_batch(self, address_pairs, batch_size=32):
        all_results = []
        for i in range(0, len(address_pairs), batch_size):
            batch = address_pairs[i : i + batch_size]
            full_addresses = []
            for p1, p2 in batch:
                parts = [p.strip() for p in (p1, p2) if p.strip()]
                full_addresses.append(" ".join(parts) if parts else "")

            encoded = self.tokenizer(
                full_addresses,
                padding=True,
                truncation=True,
                max_length=self.max_len,
                return_offsets_mapping=True,
                return_tensors="pt"
            )

            input_ids = encoded["input_ids"].to(self.device)
            attention_mask = encoded["attention_mask"].to(self.device)
            batch_offsets = encoded["offset_mapping"].tolist()

            try:
                with torch.no_grad():
                    prediction_ids_batch, emissions_batch = self.model(
                        input_ids=input_ids, attention_mask=attention_mask
                    )
            except Exception as e:
                all_results.extend([("", "", {}, {}, 0.0, f"ERROR: {str(e)}")] * len(batch))
                continue

            for idx, address_str in enumerate(full_addresses):
                if not address_str:
                    all_results.append(("", "", {}, {}, 0.0, "EMPTY_INPUT"))
                    continue

                char_tags = ["O"] * len(address_str)
                char_confs = [0.0] * len(address_str)
                offsets = batch_offsets[idx]
                prediction_ids = prediction_ids_batch[idx]
                token_probs = torch.softmax(emissions_batch[idx], dim=-1)

                for j, tag_id in enumerate(prediction_ids):
                    start, end = offsets[j]
                    if start == end: continue

                    if isinstance(tag_id, int):
                        tag = self.config.id2label[tag_id]
                        conf = token_probs[j, tag_id].item()
                    else:
                        tag = self.config.id2label[str(tag_id)]
                        conf = token_probs[j, int(tag_id)].item()

                    for c in range(start, end):
                        if char_tags[c] == "O":
                            char_tags[c] = tag
                            char_confs[c] = conf

                parsed_entities = []
                for match in re.finditer(r"([a-zA-Z0-9]+|[\u4e00-\u9fff]|[^\s])(\s*)", address_str):
                    token_str = match.group(1)
                    trailing_space = match.group(2)
                    start_idx = match.start(1)

                    tag = char_tags[start_idx]
                    conf = char_confs[start_idx]
                    entity_group = tag.replace("B-", "").replace("I-", "")
                    if tag == "O": entity_group = "O"

                    parsed_entities.append({
                        "entity_group": entity_group,
                        "word": token_str + trailing_space,
                        "conf": conf
                    })

                extracted_raw, conf_raw = self._extract_3d_components(parsed_entities)
                extracted_mapped, conf_mapped = self._map_extracted_to_standard_keys(extracted_raw, conf_raw)
                overall_conf, used_cores = self._compute_situation_aware_confidence(extracted_mapped, conf_mapped)
                split_result = self._split_address(extracted_mapped, address_str, parsed_entities)

                all_results.append((
                    split_result["line1"], split_result["line2"],
                    extracted_mapped, conf_mapped, overall_conf, "SUCCESS"
                ))

        return all_results

    def parse(self, address_part1, address_part2=""):
        result_list = self.parse_batch([(address_part1, address_part2)], batch_size=1)
        return result_list[0]

# ==========================================
# TESTING SCRIPT
# ==========================================
if __name__ == "__main__":
    MODEL_DIR = "./xlm_roberta_large_checkpointV2"

    print(f"Loading model from {MODEL_DIR}...")
    try:
        parser = HKAddressParser(model_path=MODEL_DIR)

        test_cases = [
            ("27LD, Block 5, Hemera, Lohas Park, Lohas Road 1, TKO, HK", ""),
            ("21A / 5th Fl., Metroplaza Tower A, Nº 223 Hing Fong Road, Kwai Fong, N.T.", ""),
            ("21A / 5th Fl., Metroplaza Tower 1, 223 Nº Hing Fong Road, Kwai Fong, N.T.", ""),
            ("21A / 5th Fl., Metroplaza Tower A, NW 223 Hing Fong Road, Kwai Fong, N.T.", ""),
            ("21A / 5th Fl., Metroplaza Tower 1, 223 NW Hing Fong Road, Kwai Fong, N.T.", ""),
            ("長洲東灣東堤小築彌敦道一百二十三號12座H室5樓", ""),
            ("深水埗白田街123至125號白田邨1号楼, ４０４室Part 1", ""),
            ("深水埗中正路二段123巷45弄67號白田邨1號樓廿樓, ４０４室Part 1", ""),
            ("深水埗南京路123号之2甲白田邨第1幢第5樓", ""),
            ("深水埗白田街白田邨A幢１st flor, ４０４室Part 1", ""),
            ("馬鞍山西沙路港鐵站迎海1座15樓A栋, ４０４室Part 1", ""),
            ("深水埗白田街白田邨壹號樓壹樓, ４０４室Part 1", ""),
            ("深水埗白田街白田邨一號樓一樓, ４０４室Part 1", ""),
            ("深水埗白田街白田邨第1幢第5樓", ""),
            ("香港仔中心香港仔大道港興閣", "第88座第五樓"),
            ("香港 西灣河 東苑 Block Number 8 Floor Two A", ""),
            ("香港 西灣河 東苑 Storey 4 B", ""),
            ("room8100, fu keng house, tai wo hau estate Second Flr", ""),
            ("大埔中心 座数: 1 二字樓 b", ""),
            ("大澳丈量約份第310約First Floor b", ""),
            ("Flat A, UG, Happy Mansion, 28 Lockhart Road, Wan Chai, Hong Kong", ""),
            ("Unit 7B, M/F, Harbour View Tower, 88 Gloucester Road, Causeway Bay", ""),
            ("Room 1503, LG1, Sunshine Court, 156 Nathan Road, Tsim Sha Tsui, Kowloon", ""),
            ("9C, 5-F, The Grandiose, 9 Tong Chun Street, Tseung Kwan O, New Territories", ""),
            ("Flat 3, Fifth Floor,Block Number 8, Parkview Mansion, 88 Tai Hang Road, Tai Hang, Hong Kong Island", ""),
            ("21A / 5th Fl., Metroplaza T-A, 223 Hing Fong Road, Kwai Fong, N.T.", ""),
            ("1602, 5th Flr, City One Plaza, 1 Ngan Shing Street, Sha Tin, New Territories", ""),
            ("Rm 5A, Fl. 5, Bk A, Lucky Building, 45 Queen’s Road Central, Central, HK", ""),
        ]

        print("\nRunning BATCH Test:\n" + "=" * 70)
        batch_results = parser.parse_batch(test_cases, batch_size=2)

        for i, (a1, a2) in enumerate(test_cases):
            l1, l2, tags, confs, overall, status = batch_results[i]

            print(f"Input 1 : '{a1}'")
            print(f"Input 2 : '{a2}'")
            print(f"Line 1  : {l1}")
            print(f"Line 2  : {l2}")
            print(f"Tags    : {tags}")
            print(f"Confs   : { {k: round(v, 4) for k, v in confs.items()} }")
            print(f"Overall : {overall:.6f}   (situation-aware product of core tags)")
            print(f"Status  : {status}")
            print("-" * 70)

    except FileNotFoundError as e:
        print(f"❌ Initialization Failed: {e}")
        print("Please ensure your model directory exists before running the test cases.")

Loading model from ./xlm_roberta_large_checkpointV2...

🔍 Scanning available GPUs safely via nvidia-smi...
DEBUG: Using Device -> cuda:3


Loading weights: 0it [00:00, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: ./xlm_roberta_large_checkpointV2
Key                                                           | Status     | 
--------------------------------------------------------------+------------+-
bert.encoder.layer.{0...23}.attention.self.query.weight       | UNEXPECTED | 
bert.encoder.layer.{0...23}.attention.output.dense.weight     | UNEXPECTED | 
bert.encoder.layer.{0...23}.attention.output.LayerNorm.bias   | UNEXPECTED | 
bert.encoder.layer.{0...23}.attention.self.key.weight         | UNEXPECTED | 
bert.encoder.layer.{0...23}.attention.self.value.bias         | UNEXPECTED | 
bert.embeddings.LayerNorm.bias                                | UNEXPECTED | 
bert.encoder.layer.{0...23}.attention.self.query.bias         | UNEXPECTED | 
bert.encoder.layer.{0...23}.output.LayerNorm.weight           | UNEXPECTED | 
bert.encoder.layer.{0...23}.output.dense.bias                 | UNEXPECTED | 
bert.encoder.layer.{0...23}.attention.output.LayerNorm.weigh

📦 Loading weights from ./xlm_roberta_large_checkpointV2/pytorch_model.bin ...


Loading weights: 100%|██████████| 396/396 [00:00<00:00, 478.12tensor/s, loaded=396]


✅ Loaded 396/396 tensors | missing=0 unexpected=0

Running BATCH Test:
Input 1 : '27LD, Block 5, Hemera, Lohas Park, Lohas Road 1, TKO, HK'
Input 2 : ''
Line 1  : 27LD Block 5 Hemera
Line 2  : Lohas Park, Lohas Road 1, TKO, HK
Tags    : {'floor': '27LD', 'block': 'Block 5', 'phase': 'Hemera', 'estate_name': 'Lohas Park', 'street_name': 'Lohas Road ', 'building_number': '1', 'sub_district': 'TKO', 'region': 'HK'}
Confs   : {'floor': 1.0, 'block': 0.8811, 'phase': 0.8794, 'estate_name': 0.7753, 'street_name': 0.9607, 'building_number': 0.8709, 'sub_district': 0.9875, 'region': 0.9998}
Overall : 0.600710   (situation-aware product of core tags)
Status  : SUCCESS
----------------------------------------------------------------------
Input 1 : '21A / 5th Fl., Metroplaza Tower A, Nº 223 Hing Fong Road, Kwai Fong, N.T.'
Input 2 : ''
Line 1  : 21A 5th Fl. Metroplaza Tower A
Line 2  : Nº 223 Hing Fong Road, Kwai Fong, N.T.
Tags    : {'flat': '21A ', 'floor': '5th Fl.', 'building_name': 'Metropl